# Final Model Evaluation

The test set has already been accessed in prior analysis, and this notebook already contains test metrics. It must not be described as an untouched, first-time, or blind holdout. The saved test results are historical results for the artifact versions loaded when they were produced. This notebook now points to explicit MLflow run versions; the historical MLP and LSTM test metrics do not apply to the newer run-specific artifacts. Editing these paths does not recalculate results. Any later execution of the evaluation cells would be another evaluation on the previously used test set.

The evaluation will focus on the models selected during validation:

- Random Forest
- MLP
- LSTM

Average Precision (AP) will remain the primary evaluation metric because of the severe class imbalance in the fraud detection problem.

Precision, Recall, F1-score, ROC-AUC, and confusion matrices will be reported as secondary evaluation metrics.

No model or threshold will be modified based on the final test results.

## 1. Selected Models

The candidate models for final evaluation were selected using validation performance only.

The historical notebook validation reference showed Random Forest leading at Average Precision (AP) 0.858000, followed by MLP at 0.857221 and LSTM at 0.848276. The legacy root-level artifacts recorded reproduced validation AP values of Random Forest 0.857966, MLP 0.846886, and LSTM 0.846636. The explicitly selected MLflow runs below record validation AP values of Random Forest 0.857966367810208, MLP 0.8551813692570064, and LSTM 0.8538918557190249. The Random Forest model bytes match the registered run artifact; the legacy MLP and LSTM weights differ from the selected run-specific weights. Existing test metrics remain attached to the legacy artifacts and are not results for the newer MLP or LSTM versions.

The historical MLP result was therefore retained as a competitive tabular deep learning representative.

The historical LSTM result was retained as the representative temporal deep learning model.

The GRU, XGBoost, Logistic Regression, LightGBM, and Autoencoder will not be evaluated on the final test set because they did not meet the selection criteria established during validation.

The selected models are therefore:

1. Random Forest
2. MLP
3. LSTM

Random Forest remains selected because it led on validation AP. MLP and LSTM are retained as competitive representatives, and test results do not retroactively change model selection. The existing stored test results are not recalculated by this notebook edit. The 0.5 threshold is a standard diagnostic threshold, not a production threshold.

In [ ]:
# Locate the repository root before importing the shared project module.
import sys
from pathlib import Path
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "notebooks").is_dir()
        and (candidate / "data").is_dir()
        and (candidate / "models").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the Bank-Fraud-Detection project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the shared canonical inference registry and scoring interface.
from src.inference import CANONICAL_MODELS, EXPECTED_FEATURES, load_model, predict_scores

# Import numerical/dataframe libraries and the existing evaluation metrics.
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
EXPECTED_FEATURES = list(EXPECTED_FEATURES)

print(f"Models directory: {MODELS_DIR.resolve()}")
print("Canonical model versions are defined by src.inference.CANONICAL_MODELS.")


In [ ]:
# Load the selected canonical model versions through the shared interface.
# The loader verifies local hashes, feature/preprocessing metadata, and the
# appropriate training-run or artifact-import lineage record.
SELECTED_MODELS = ("random_forest", "mlp", "lstm")
loaded_models = {name: load_model(name) for name in SELECTED_MODELS}

random_forest_metadata = loaded_models["random_forest"].metadata
mlp_metadata = loaded_models["mlp"].metadata
lstm_metadata = loaded_models["lstm"].metadata
random_forest_artifact_path = loaded_models["random_forest"].artifact_path
mlp_artifact_path = loaded_models["mlp"].artifact_path
lstm_artifact_path = loaded_models["lstm"].artifact_path
random_forest_model = loaded_models["random_forest"].estimator
mlp_model = loaded_models["mlp"].estimator
lstm_model = loaded_models["lstm"].estimator

for model_name, loaded_model in loaded_models.items():
    spec = loaded_model.spec
    print(f"{model_name}: canonical artifact {spec.artifact_path}")
    if spec.training_run_id is not None:
        print(f"  Training MLflow run: {spec.training_run_id}")
    else:
        print(f"  Artifact-import MLflow run (not a training run): {spec.artifact_import_run_id}")
    print(f"  Metadata pipeline ID: {spec.pipeline_run_id}")
    print(f"  Feature count: {len(loaded_model.metadata['features'])}")


In [ ]:
# The shared loader reconstructs training-defined architectures, loads each
# state dictionary with strict=True, and places PyTorch models in evaluation mode.
assert not mlp_model.training
assert not lstm_model.training


## 2. Final Test-Set Integrity Checks

The test set was accessed in prior analysis and final metrics are already stored in this notebook. Executing this section would access that same test split again; it is not a first-time or blind evaluation. The previously stored test outputs correspond to the legacy artifacts and must not be attributed to the newer MLP or LSTM run versions configured above. This edit does not execute or recalculate any evaluation.

In [4]:
# Load only the test feature matrix and target for final-evaluation readiness checks.
X_test = pd.read_parquet(DATA_DIR / "X_test.parquet")
y_test = pd.read_parquet(DATA_DIR / "y_test.parquet").squeeze("columns")

# Compare the test feature order against each selected model's stored feature contract.
selected_feature_lists = {
    "Random Forest": random_forest_metadata["features"],
    "MLP": mlp_metadata["features"],
    "LSTM": lstm_metadata["features"],
}
expected_features = random_forest_metadata["features"]
assert len(expected_features) == 32, "Selected model metadata must contain 32 features."
assert all(features == expected_features for features in selected_feature_lists.values()), \
    "Selected model feature lists do not match one another."
assert X_test.shape[1] == 32, f"Expected 32 test features, got {X_test.shape[1]}."
assert X_test.columns.tolist() == expected_features, \
    "Test feature order does not match the selected model metadata."

# Confirm that the chronological 70/15/15 split produced its expected test size.
EXPECTED_TEST_OBSERVATIONS = 42_722
assert len(X_test) == EXPECTED_TEST_OBSERVATIONS, \
    f"Expected {EXPECTED_TEST_OBSERVATIONS} test rows, got {len(X_test)}."
assert len(y_test) == EXPECTED_TEST_OBSERVATIONS, \
    f"Expected {EXPECTED_TEST_OBSERVATIONS} test labels, got {len(y_test)}."
assert len(X_test) == len(y_test), "Test features and labels have different row counts."

# Report test-set composition and data-integrity checks without making predictions.
test_class_counts = y_test.value_counts().reindex([0, 1], fill_value=0).sort_index()
test_fraud_rate = float((y_test == 1).mean())
test_feature_missing_values = int(X_test.isna().to_numpy().sum())
test_target_missing_values = int(y_test.isna().sum())
assert test_feature_missing_values == 0, "Test features contain missing values."
assert test_target_missing_values == 0, "Test target contains missing values."

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Expected test observations: {EXPECTED_TEST_OBSERVATIONS}")
print(f"Feature count: {X_test.shape[1]}")
print(f"Feature order matches all selected model metadata: {X_test.columns.tolist() == expected_features}")
print("Test class counts (0 = legitimate, 1 = fraud):")
print(test_class_counts.to_string())
print(f"Test fraud rate: {test_fraud_rate:.6%}")
print(f"Missing values in X_test: {test_feature_missing_values}")
print(f"Missing values in y_test: {test_target_missing_values}")
print(f"Feature and target row counts match: {len(X_test) == len(y_test)}")
print("No predictions or test performance metrics have been calculated.")

X_test shape: (42722, 32)
y_test shape: (42722,)
Expected test observations: 42722
Feature count: 32
Feature order matches all selected model metadata: True
Test class counts (0 = legitimate, 1 = fraud):
Class
0    42670
1       52
Test fraud rate: 0.121717%
Missing values in X_test: 0
Missing values in y_test: 0
Feature and target row counts match: True
No predictions or test performance metrics have been calculated.


## 3. Final-Evaluation Fraud Probabilities

Generate fraud probabilities for the selected Random Forest, MLP, and LSTM using the test split only. Random Forest receives the original feature values; MLP and LSTM receive the same RobustScaler transformation used during their training. The LSTM uses chronological sliding windows of length 10, constructed wholly within the test split. Therefore, the first 9 test transactions do not become sequence targets, yielding 42,713 LSTM probabilities from 42,722 test transactions. No threshold or test metric is applied in this stage.

**Archived output notice:** The saved probability diagnostics below were produced before this lineage update using the root-level artifacts. They are preserved historical outputs, not predictions from the newly configured MLP and LSTM run artifacts. The Random Forest model bytes are identical to its configured run artifact; the legacy MLP and LSTM weights are different. Re-running these cells would generate a new evaluation on the previously used test set.

In [5]:
# Generate scores using the shared inference implementation and already loaded test
# frame. This cell does not refit preprocessing or calculate evaluation metrics.
random_forest_result = predict_scores(loaded_models["random_forest"], X_test)
mlp_result = predict_scores(loaded_models["mlp"], X_test)
lstm_result = predict_scores(loaded_models["lstm"], X_test)

random_forest_fraud_probabilities = random_forest_result.scores
mlp_fraud_probabilities = mlp_result.scores
lstm_fraud_probabilities = lstm_result.scores

sequence_length = int(lstm_metadata["configuration"]["sequence_length"])
SEQUENCE_LENGTH = sequence_length  # Preserve the downstream label-alignment variable.
assert sequence_length == 10
assert random_forest_result.row_indices.equals(X_test.index)
assert mlp_result.row_indices.equals(X_test.index)
assert lstm_result.row_indices.equals(X_test.index[sequence_length - 1:])

# Keep the notebook's existing expected test-row counts and probability checks.
expected_prediction_lengths = {
    "Random Forest": 42_722,
    "MLP": 42_722,
    "LSTM": 42_713,
}
fraud_probability_predictions = {
    "Random Forest": random_forest_fraud_probabilities,
    "MLP": mlp_fraud_probabilities,
    "LSTM": lstm_fraud_probabilities,
}
for model_name, probabilities in fraud_probability_predictions.items():
    assert probabilities.shape == (expected_prediction_lengths[model_name],)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0.0) & (probabilities <= 1.0)).all()

# Display only basic diagnostics; the existing metric cell remains separate.
for model_name, probabilities in fraud_probability_predictions.items():
    print(f"{model_name}:")
    print(f"  shape: {probabilities.shape}")
    print(f"  minimum probability: {probabilities.min():.8f}")
    print(f"  maximum probability: {probabilities.max():.8f}")
    print(f"  mean probability: {probabilities.mean():.8f}")
    print(f"  finite values: {int(np.isfinite(probabilities).sum())}")


Random Forest:
  shape: (42722,)
  minimum probability: 0.00000000
  maximum probability: 0.96500000
  mean probability: 0.00138196
  finite values: 42722
MLP:
  shape: (42722,)
  minimum probability: 0.00000000
  maximum probability: 1.00000000
  mean probability: 0.01670110
  finite values: 42722
LSTM:
  shape: (42713,)
  minimum probability: 0.00036882
  maximum probability: 0.99932766
  mean probability: 0.02897878
  finite values: 42713


## 4. Final Test-Set Metrics

Evaluate only the selected Random Forest, MLP, and LSTM predictions already generated above. Average Precision and ROC-AUC use continuous fraud probabilities. Precision, Recall, F1-score, and the confusion matrix use the standard descriptive threshold of 0.5; no test-set threshold optimization or operating-threshold selection is performed.

The LSTM evaluation uses labels from the final transaction of each length-10 test sequence. The first 9 original test transactions are excluded because they cannot form a complete sequence.

**Archived metric notice:** The saved table below is the prior test evaluation of the root-level legacy artifacts. Its MLP and LSTM test metrics do not describe the newer run-specific weights configured above. The table is retained as historical evidence and has not been recalculated.

In [6]:
# Align each already-generated probability array with its corresponding test labels.
y_test_full = y_test.to_numpy(dtype=np.int64, copy=True)
y_test_lstm = y_test.iloc[SEQUENCE_LENGTH - 1:].to_numpy(dtype=np.int64, copy=True)
assert len(y_test_full) == 42_722
assert len(y_test_lstm) == 42_713
assert len(y_test_full) == len(random_forest_fraud_probabilities) == len(mlp_fraud_probabilities)
assert len(y_test_lstm) == len(lstm_fraud_probabilities)

# Calculate ranking metrics from continuous probabilities and threshold metrics at 0.5.
evaluation_inputs = [
    ("Random Forest", random_forest_fraud_probabilities, y_test_full),
    ("MLP", mlp_fraud_probabilities, y_test_full),
    ("LSTM", lstm_fraud_probabilities, y_test_lstm),
]
final_test_rows = []
for model_name, probabilities, targets in evaluation_inputs:
    binary_predictions = (probabilities >= 0.5).astype(np.int64)
    matrix = confusion_matrix(targets, binary_predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    final_test_rows.append({
        "Model": model_name,
        "AP": average_precision_score(targets, probabilities),
        "Precision @ 0.5": precision_score(targets, binary_predictions, zero_division=0),
        "Recall @ 0.5": recall_score(targets, binary_predictions, zero_division=0),
        "F1 @ 0.5": f1_score(targets, binary_predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(targets, probabilities),
        "Confusion Matrix [ [TN, FP], [FN, TP] ]": f"[[{tn}, {fp}], [{fn}, {tp}]]",
    })

# Present the final test results; no threshold is selected from these results.
final_test_metrics = pd.DataFrame(final_test_rows).set_index("Model")
print(final_test_metrics.to_string(float_format=lambda value: f"{value:.6f}"))

                    AP  Precision @ 0.5  Recall @ 0.5  F1 @ 0.5  ROC-AUC Confusion Matrix [ [TN, FP], [FN, TP] ]
Model                                                                                                           
Random Forest 0.752371         0.972222      0.673077  0.795455 0.937687                  [[42669, 1], [17, 35]]
MLP           0.753003         0.167347      0.788462  0.276094 0.964874                [[42466, 204], [11, 41]]
LSTM          0.740742         0.106952      0.769231  0.187793 0.982393                [[42327, 334], [12, 40]]


## 5. Historical Test Evaluation Interpretation

The saved table and interpretation below are preserved results from the earlier evaluation of the root-level legacy artifacts. Random Forest is byte-identical to the currently configured run artifact; the legacy MLP and LSTM weights differ from their newer configured runs. The MLP and LSTM test results below therefore must not be attributed to the current run-specific weights.

### Generalization from validation to test

All three legacy artifacts had lower Average Precision (AP) on the test set than their own reproduced validation AP: Random Forest decreased from 0.857966 to 0.752371, MLP from 0.846886 to 0.753003, and LSTM from 0.846636 to 0.740742. Validation ranking performance therefore did not fully carry over to the previously unseen transactions in that evaluation. These results alone do not establish why the reduction occurred.

### Model results

Random Forest was the validation leader in the historical notebook comparison at AP 0.858000; its root-level artifact metadata recorded reproduced validation AP 0.857966. Its test AP was 0.752371. At the standard threshold of 0.5, it achieved 97.22% precision, 67.31% recall, and 79.55% F1, with one false positive and 35 of the 52 fraud cases detected. This is strong threshold-specific behavior when limiting false alerts is important. The threshold was not optimized on the test set.

MLP achieved test AP 0.753003, just 0.000632 above Random Forest. At threshold 0.5, MLP had higher recall (78.85%) but much lower precision (16.73%), with 204 false positives. This reflects a different precision-recall trade-off and does not make MLP superior overall.

LSTM achieved test AP 0.740742 and the highest ROC-AUC among these models at 0.982393; AP remains the project's primary ranking metric. At threshold 0.5, LSTM generated 334 false positives, substantially more than Random Forest. Its sequences represent windows of 10 globally chronological transactions. Because the dataset has no customer, account, merchant, or device identifiers, these windows do not represent individual entities' transaction histories. The first 9 test transactions were excluded from LSTM target evaluation because they cannot form a complete sequence.

### Final model and threshold

Random Forest remains the selected model based on validation performance. The historical test results below are retained for reporting and do not retroactively change that validation-based selection. Although MLP's test AP is marginally higher, the models are essentially tied on test AP, while Random Forest has substantially higher threshold-0.5 precision and F1 and far fewer false positives.

The 0.5 threshold is only the standard diagnostic threshold used for this evaluation. The 0.5 threshold is not a production threshold. No production operating threshold has been selected or optimized using the test set. The test set had been accessed in prior analysis and is not an untouched or blind holdout. These stored metrics are historical results for the artifact versions described above; any future evaluation would reuse the previously accessed split.